# WebScraping API da Betfair

### Instalando a biblioteca betfairlightweight

In [2]:
# pip install betfairlightweight

### Importando as Bibliotecas

In [3]:
from futpythontrader_DEMO import *

import datetime
import pandas as pd
import pytz
import betfairlightweight

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### Fazendo login na API da Betfair

In [9]:
# username = "EMAIL"
# pw = "SENHA"
# app_key = "APP_KEY"

# Iniciando a conexão com a API da Betfair
trading = betfairlightweight.APIClient(username, pw, app_key=app_key, cert_files=('./Betfair/Betfair.crt','./Betfair/Betfair.pem'))
trading.login()

<LoginResource>

### Criando o Filtro de Mercado

In [14]:
dias = 1

filtros_mercado = betfairlightweight.filters.market_filter(
    event_type_ids=['1'], 
    market_start_time={
        'to': (datetime.datetime.utcnow() + datetime.timedelta(days=dias)).strftime("%Y-%m-%dT%TZ")
    }
)

### Listando Eventos e Competições

In [15]:
eventos_futebol = trading.betting.list_events(filter=filtros_mercado)
competicoes_futebol = trading.betting.list_competitions(filter=filtros_mercado)

### Criando um Dicionário para Competições

In [16]:
competitions = {event_object.competition.name: event_object.competition.id 
                for event_object in competicoes_futebol}

### Criando e Organizando o DataFrame

In [17]:
df = pd.DataFrame(list(competitions.items()), columns=['League', 'Id']).sort_values('League').reset_index(drop=True)
df

,League,Id
0,Albanian Superliga,11627247
1,Algerian Ligue 1,4431442
2,Argentinian Copa de la Liga Profesional,12225162
3,Argentinian Primera B Metropolitana,11633637
4,Argentinian Primera B Nacional,803237
5,Argentinian Primera C,7511144
6,Argentinian Torneo A,12103652
7,Australian A-League Men,12117172
8,Austrian Bundesliga,10479956
9,Austrian Erste Liga,9


### Obter IDs das Competições do DataFrame

In [18]:
competition_ids = df['Id'].to_list()
competition_ids

['11627247',
 '4431442',
 '12225162',
 '11633637',
 '803237',
 '7511144',
 '12103652',
 '12117172',
 '10479956',
 '9',
 '12206014',
 '89979',
 '11717188',
 '12007003',
 '3127357',
 '12185349',
 '12181991',
 '12172193',
 '12184051',
 '12185073',
 '12184071',
 '12185071',
 '7422970',
 '12184232',
 '12148223',
 '15',
 '987643',
 '12149039',
 '744098',
 '3085749',
 '844197',
 '856134',
 '2079376',
 '17',
 '3992967',
 '12204204',
 '862638',
 '25',
 '23',
 '12222436',
 '11',
 '9404054',
 '9882094',
 '803690',
 '7129730',
 '35',
 '37',
 '43',
 '10932509',
 '873203',
 '12216175',
 '11804597',
 '55',
 '12206060',
 '11591693',
 '59',
 '61',
 '11620783',
 '63',
 '318343',
 '65',
 '67',
 '12267123',
 '6566654',
 '6812603',
 '12202271',
 '833990',
 '1842928',
 '4556576',
 '12209556',
 '6251219',
 '12203971',
 '822165',
 '81',
 '12199689',
 '12199513',
 '3785366',
 '89',
 '1062024',
 '4802285',
 '12200652',
 '12216489',
 '7616742',
 '6908995',
 '12216976',
 '879221',
 '4515700',
 '1517121',
 '132739

### Criando Listas Vazias para Armazenar Informações

In [19]:
eventos_list = []
mercado_evento_id = []
nomes_competicoes = []

In [20]:
competition_ids = ['59','10932509'] 

### Processar Cada Competição

In [21]:
for id_competicao in competition_ids:
    try:
        competicoes = trading.betting.list_competitions(
            filter=betfairlightweight.filters.market_filter(competition_ids=[id_competicao])
        )
        nome_competicao = competicoes[0].competition.name if competicoes else 'Competição não encontrada'
        nomes_competicoes.append(nome_competicao)
    except Exception as e:
        print(f"Erro ao processar competição {id_competicao}: {e}")
        continue

### Configurando Datas e Horários

In [22]:
fuso_horario = 'America/Sao_Paulo' 
agora = datetime.datetime.now(pytz.timezone(fuso_horario)) 
data_atual = agora.date() 
data_futura = data_atual + datetime.timedelta(days=dias)

inicio_do_dia = datetime.datetime.combine(data_atual, datetime.time.min).astimezone(pytz.timezone(fuso_horario))
fim_do_dia = datetime.datetime.combine(data_futura, datetime.time.max).astimezone(pytz.timezone(fuso_horario))

### Definindo os Mercados Desejados

In [23]:
mercados_desejados = [
    "Match Odds", "First Half Goals 0.5", "First Half Goals 1.5", "First Half Goals 2.5",
    "Over/Under 0.5 Goals", "Over/Under 1.5 Goals", "Over/Under 2.5 Goals", "Over/Under 3.5 Goals",
    "Over/Under 4.5 Goals", "Over/Under 5.5 Goals", "Over/Under 6.5 Goals", "Over/Under 7.5 Goals",
    "Over/Under 8.5 Goals", "Both teams to Score?", "Correct Score",
    ]

### Configurando o Filtro de Preços

In [24]:
price_filter = betfairlightweight.filters.price_projection(
    price_data=['EX_BEST_OFFERS'],
    ex_best_offers_overrides=betfairlightweight.filters.ex_best_offers_overrides(best_prices_depth=1)
)

### Iteração com a API da Betfair

In [25]:
# Processando Cada Competição
for competition_id in competition_ids:
    nome_competicao = nomes_competicoes[competition_ids.index(competition_id)]
    
    filtros_mercado = betfairlightweight.filters.market_filter(
        event_type_ids=['1'], 
        market_start_time={'from': inicio_do_dia.strftime('%Y-%m-%dT%H:%M:%SZ'), 'to': fim_do_dia.strftime('%Y-%m-%dT%H:%M:%SZ')},
        competition_ids=[competition_id]
    )
    
    # Obtendo e Filtrando Eventos por Competição
    try:
        eventos_futebol = trading.betting.list_events(filter=filtros_mercado)
    except Exception as e:
        print(f"Erro ao listar eventos para competição {competition_id}: {e}")
        continue
    
    # Processando Cada Evento
    for obj_evento in eventos_futebol:
        id_evento = obj_evento.event.id

        filtro_catalogo_mercados = betfairlightweight.filters.market_filter(event_ids=[id_evento])
        
        # Filtrando e Obtendo Catálogos de Mercado
        try:
            catalogos_mercado = trading.betting.list_market_catalogue(
                filter=filtro_catalogo_mercados, max_results='100', sort='FIRST_TO_START', market_projection=['RUNNER_METADATA']
            )
        except Exception as e:
            print(f"Erro ao listar catálogo de mercado para evento {id_evento}: {e}")
            continue

        market_ids = {}
        odds_dict = {}
        selection_ids = {}
        
        # Processando Cada Mercado no Catálogo
        for mercado in catalogos_mercado:
            if mercado.market_name in mercados_desejados:
                market_ids['IDMercado_' + mercado.market_name.replace(" ", "_").replace(".", "")] = mercado.market_id

                try:
                    market_books = trading.betting.list_market_book(market_ids=[mercado.market_id], price_projection=price_filter)
                except Exception as e:
                    print(f"Erro ao listar book de mercado {mercado.market_id}: {e}")
                    continue
                
                # Obtendo e Processando Informações do Mercado
                for idx, runner in enumerate(market_books[0].runners):
                    try:
                        back_price = runner.ex.available_to_back[0].price if runner.ex.available_to_back else 1.01
                        lay_price = runner.ex.available_to_lay[0].price if runner.ex.available_to_lay else 1000
                        
                        if mercado.market_name == "Match Odds":
                            selection_key = None
                            if idx == 0:
                                nome_coluna = "Match_Odds_Home"
                                selection_key = 'ID_Home'
                            elif idx == 1:
                                nome_coluna = "Match_Odds_Away"
                                selection_key = 'ID_Away'
                            elif idx == 2:
                                nome_coluna = "Match_Odds_Draw"
                                selection_key = 'ID_Draw'
                            if selection_key:
                                selection_ids[selection_key] = runner.selection_id
                        else:
                            nome_coluna = f"{mercado.market_name.replace(' ', '_')}_{runner.selection_id}"

                        odds_dict[nome_coluna] = {'Back': back_price, 'Lay': lay_price}
                    except IndexError:
                        odds_dict[nome_coluna] = {'Back': 1.01, 'Lay': 1000}

        nome_evento = obj_evento.event.name
        times = nome_evento.split(" v ")
        if len(times) != 2:
            print(f"Formato inesperado do nome do evento: {nome_evento}")
            continue

        data_hora = obj_evento.event.open_date.replace(tzinfo=datetime.timezone.utc).astimezone(tz=None)
        
        # Criando e Atualizando um Dicionário de Eventos
        evento_dict = {
            'ID_Evento': id_evento,
            'League': nome_competicao,
            'Date': data_hora.strftime('%Y-%m-%d'),
            'Time': data_hora.strftime('%H:%M:%S'),
            'Home': times[0],
            'Away': times[1]
        }

        evento_dict.update(odds_dict)
        evento_dict.update(market_ids)
        evento_dict.update(selection_ids)
        eventos_list.append(evento_dict)

# Criando uma DataFrame com os Eventos
planilha_eventos_futebol = pd.DataFrame(eventos_list)

Formato inesperado do nome do evento: Daily Goals


In [26]:
planilha_eventos_futebol

,ID_Evento,League,Date,Time,Home,Away,Over/Under_3.5_Goals_1222344,Over/Under_3.5_Goals_1222345,Over/Under_7.5_Goals_1485572,Over/Under_7.5_Goals_1485573,Both_teams_to_Score?_30246,Both_teams_to_Score?_110503,Over/Under_8.5_Goals_2407528,Over/Under_8.5_Goals_2407529,First_Half_Goals_2.5_47972,First_Half_Goals_2.5_47973,Over/Under_4.5_Goals_1222347,Over/Under_4.5_Goals_1222346,Over/Under_2.5_Goals_47972,Over/Under_2.5_Goals_47973,Over/Under_1.5_Goals_1221385,Over/Under_1.5_Goals_1221386,Correct_Score_1,Correct_Score_4,Correct_Score_9,Correct_Score_16,Correct_Score_2,Correct_Score_3,Correct_Score_8,Correct_Score_15,Correct_Score_5,Correct_Score_6,Correct_Score_7,Correct_Score_14,Correct_Score_10,Correct_Score_11,Correct_Score_12,Correct_Score_13,Correct_Score_9063254,Correct_Score_9063255,Correct_Score_9063256,Match_Odds_Home,Match_Odds_Away,Match_Odds_Draw,Over/Under_6.5_Goals_2542448,Over/Under_6.5_Goals_2542449,Over/Under_5.5_Goals_1485567,Over/Under_5.5_Goals_1485568,First_Half_Goals_0.5_5851482,First_Half_Goals_0.5_5851483,First_Half_Goals_1.5_1221385,First_Half_Goals_1.5_1221386,Over/Under_0.5_Goals_5851482,Over/Under_0.5_Goals_5851483,IDMercado_Over/Under_35_Goals,IDMercado_Over/Under_75_Goals,IDMercado_Both_teams_to_Score?,IDMercado_Over/Under_85_Goals,IDMercado_First_Half_Goals_25,IDMercado_Over/Under_45_Goals,IDMercado_Over/Under_25_Goals,IDMercado_Over/Under_15_Goals,IDMercado_Correct_Score,IDMercado_Match_Odds,IDMercado_Over/Under_65_Goals,IDMercado_Over/Under_55_Goals,IDMercado_First_Half_Goals_05,IDMercado_First_Half_Goals_15,IDMercado_Over/Under_05_Goals,ID_Home,ID_Away,ID_Draw
0,33118344,German Bundesliga,2024-04-06,10:30:00,Mainz,SV Darmstadt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,33118354,German Bundesliga,2024-04-07,12:30:00,Wolfsburg,Mgladbach,"{'Back': 1.55, 'Lay': 1.59}","{'Back': 2.72, 'Lay': 2.82}","{'Back': 1.01, 'Lay': 1.02}","{'Back': 70.0, 'Lay': 1000.0}","{'Back': 1.66, 'Lay': 1.73}","{'Back': 2.38, 'Lay': 2.5}","{'Back': 1.01, 'Lay': 1.01}","{'Back': 190.0, 'Lay': 1000.0}","{'Back': 1.15, 'Lay': 1.2}","{'Back': 6.0, 'Lay': 7.8}","{'Back': 1.22, 'Lay': 1.24}","{'Back': 5.2, 'Lay': 5.5}","{'Back': 2.38, 'Lay': 2.42}","{'Back': 1.71, 'Lay': 1.73}","{'Back': 5.0, 'Lay': 5.8}","{'Back': 1.21, 'Lay': 1.25}","{'Back': 18.5, 'Lay': 19.5}","{'Back': 18.0, 'Lay': 20.0}","{'Back': 30.0, 'Lay': 34.0}","{'Back': 60.0, 'Lay': 85.0}","{'Back': 11.0, 'Lay': 12.0}","{'Back': 8.6, 'Lay': 9.2}","{'Back': 15.5, 'Lay': 16.5}","{'Back': 40.0, 'Lay': 44.0}","{'Back': 12.0, 'Lay': 13.5}","{'Back': 9.6, 'Lay': 10.5}","{'Back': 15.5, 'Lay': 16.5}","{'Back': 42.0, 'Lay': 48.0}","{'Back': 20.0, 'Lay': 22.0}","{'Back': 16.0, 'Lay': 17.5}","{'Back': 26.0, 'Lay': 28.0}","{'Back': 55.0, 'Lay': 75.0}","{'Back': 9.2, 'Lay': 9.6}","{'Back': 30.0, 'Lay': 36.0}","{'Back': 22.0, 'Lay': 980.0}","{'Back': 1.93, 'Lay': 1.94}","{'Back': 4.2, 'Lay': 4.3}","{'Back': 4.1, 'Lay': 4.2}","{'Back': 1.02, 'Lay': 1.04}","{'Back': 26.0, 'Lay': 55.0}","{'Back': 1.08, 'Lay': 1.1}","{'Back': 11.0, 'Lay': 14.5}","{'Back': 3.4, 'Lay': 4.2}","{'Back': 1.31, 'Lay': 1.41}","{'Back': 1.52, 'Lay': 1.63}","{'Back': 2.6, 'Lay': 2.9}","{'Back': 17.5, 'Lay': 21.0}","{'Back': 1.05, 'Lay': 1.07}",1.226313910,1.226313860,1.226313861,1.226313858,1.226313859,1.226313847,1.226313846,1.226313849,1.226313871,1.226313862,1.226313869,1.226313870,1.226313866,1.226313868,1.226313867,44521.0,84649.0,58805.0
2,33118391,German Bundesliga,2024-04-07,10:30:00,Hoffenheim,Augsburg,"{'Back': 1.78, 'Lay': 1.82}","{'Back': 2.22, 'Lay': 2.3}","{'Back': 1.01, 'Lay': 1.03}","{'Back': 38.0, 'Lay': 110.0}","{'Back': 1.5, 'Lay': 1.53}","{'Back': 2.92, 'Lay': 3.0}","{'Back': 1.01, 'Lay': 1.01}","{'Back': 100.0, 'Lay': 1000.0}","{'Back': 1.17, 'Lay': 1.3}","{'Back': 4.4,

### Selecionando Colunas do DataFrame

In [27]:
df = planilha_eventos_futebol[['League','Date','Time','Home','Away',
'Match_Odds_Home','Match_Odds_Away','Match_Odds_Draw',
'First_Half_Goals_0.5_5851482','First_Half_Goals_0.5_5851483',
'First_Half_Goals_1.5_1221385','First_Half_Goals_1.5_1221386',
'First_Half_Goals_2.5_47972','First_Half_Goals_2.5_47973',
'Over/Under_0.5_Goals_5851482','Over/Under_0.5_Goals_5851483',
'Over/Under_1.5_Goals_1221385','Over/Under_1.5_Goals_1221386',
'Over/Under_2.5_Goals_47972','Over/Under_2.5_Goals_47973',
'Over/Under_3.5_Goals_1222344','Over/Under_3.5_Goals_1222345',
'Over/Under_4.5_Goals_1222347','Over/Under_4.5_Goals_1222346',
'Over/Under_5.5_Goals_1485567','Over/Under_5.5_Goals_1485568',
'Over/Under_6.5_Goals_2542448','Over/Under_6.5_Goals_2542449',
'Over/Under_7.5_Goals_1485572','Over/Under_7.5_Goals_1485573',
'Over/Under_8.5_Goals_2407528','Over/Under_8.5_Goals_2407529',
'Both_teams_to_Score?_30246','Both_teams_to_Score?_110503',
'Correct_Score_1','Correct_Score_4','Correct_Score_9','Correct_Score_16',
'Correct_Score_2','Correct_Score_3','Correct_Score_8','Correct_Score_15',
'Correct_Score_5','Correct_Score_6','Correct_Score_7','Correct_Score_14',
'Correct_Score_10','Correct_Score_11','Correct_Score_12','Correct_Score_13',
'Correct_Score_9063254','Correct_Score_9063255','Correct_Score_9063256']]

### Renomeando Colunas

In [28]:
df.columns = ['League','Date','Time','Home','Away',
'Odd_H','Odd_A','Odd_D',
'Odd_Under05_HT','Odd_Over05_HT',
'Odd_Under15_HT','Odd_Over15_HT',
'Odd_Under25_HT','Odd_Over25_HT',
'Odd_Under05_FT','Odd_Over05_FT',
'Odd_Under15_FT','Odd_Over15_FT',
'Odd_Under25_FT','Odd_Over25_FT',
'Odd_Under35_FT','Odd_Over35_FT',
'Odd_Under45_FT','Odd_Over45_FT',
'Odd_Under55_FT','Odd_Over55_FT',
'Odd_Under65_FT','Odd_Over65_FT',
'Odd_Under75_FT','Odd_Over75_FT',
'Odd_Under85_FT','Odd_Over85_FT',
'Odd_BTTS_Yes','Odd_BTTS_No',
'Odd_CS_0x0','Odd_CS_0x1','Odd_CS_0x2','Odd_CS_0x3',
'Odd_CS_1x0','Odd_CS_1x1','Odd_CS_1x2','Odd_CS_1x3',
'Odd_CS_2x0','Odd_CS_2x1','Odd_CS_2x2','Odd_CS_2x3',
'Odd_CS_3x0','Odd_CS_3x1','Odd_CS_3x2','Odd_CS_3x3',
'Odd_CS_Goleada_H','Odd_CS_Goleada_A','Odd_CS_Goleada_D']

### Preparando uma Lista de Colunas Específicas

In [30]:
colunas_para_separar = [
    'Odd_H', 'Odd_A', 'Odd_D', 'Odd_Under05_HT', 'Odd_Over05_HT', 'Odd_Under15_HT', 'Odd_Over15_HT', 
    'Odd_Under25_HT', 'Odd_Over25_HT', 'Odd_Under05_FT', 'Odd_Over05_FT', 'Odd_Under15_FT', 'Odd_Over15_FT', 
    'Odd_Under25_FT', 'Odd_Over25_FT', 'Odd_Under35_FT', 'Odd_Over35_FT', 'Odd_Under45_FT', 'Odd_Over45_FT', 
    'Odd_Under55_FT', 'Odd_Over55_FT', 'Odd_Under65_FT', 'Odd_Over65_FT', 'Odd_Under75_FT', 'Odd_Over75_FT', 
    'Odd_Under85_FT', 'Odd_Over85_FT', 'Odd_BTTS_Yes', 'Odd_BTTS_No', 'Odd_CS_0x0', 'Odd_CS_0x1', 
    'Odd_CS_0x2', 'Odd_CS_0x3', 'Odd_CS_1x0', 'Odd_CS_1x1', 'Odd_CS_1x2', 'Odd_CS_1x3', 'Odd_CS_2x0', 
    'Odd_CS_2x1', 'Odd_CS_2x2', 'Odd_CS_2x3', 'Odd_CS_3x0', 'Odd_CS_3x1', 'Odd_CS_3x2', 'Odd_CS_3x3', 
    'Odd_CS_Goleada_H', 'Odd_CS_Goleada_A', 'Odd_CS_Goleada_D'
]

### Criando um Novo DataFrame

In [31]:
jogos_do_dia = pd.DataFrame()

### Iterando Pelas Colunas do DataFrame Original e Processando Colunas Específicas

In [32]:
for coluna in df.columns:
    if coluna in colunas_para_separar:
        jogos_do_dia[coluna + '_Back'] = df[coluna].apply(lambda x: x['Back'] if isinstance(x, dict) and 'Back' in x else None)
        jogos_do_dia[coluna + '_Lay'] = df[coluna].apply(lambda x: x['Lay'] if isinstance(x, dict) and 'Lay' in x else None)
    else:
        jogos_do_dia[coluna] = df[coluna]

### Processamento do DataFrame para Hoje e Amanhã

In [34]:
def hoje():
    from datetime import date, datetime, timedelta
    dia = date.today() + timedelta(0)
    return dia

def amanha():
    from datetime import date, datetime, timedelta
    dia = date.today() + timedelta(1)
    return dia

def depois_de_amanha():
    from datetime import date, datetime, timedelta
    dia = date.today() + timedelta(2)
    return dia

def drop_reset_index(df):
    df = df.dropna()
    df = df.reset_index(drop=True)
    df.index += 1
    return df

hoje = str(hoje())

flt = jogos_do_dia.Date == hoje
df_hoje = jogos_do_dia[flt]
df_hoje = df_hoje.sort_values(['Time'])
df_hoje = drop_reset_index(df_hoje)
display(df_hoje)

amanha = str(amanha())

flt = jogos_do_dia.Date == amanha
df_amanha = jogos_do_dia[flt]
df_amanha = df_amanha.sort_values(['Time'])
df_amanha = drop_reset_index(df_amanha)
display(df_amanha)
# df_amanha.to_excel(f"Jogos_do_Dia_Betfair_Back_Lay_{amanha}.xlsx",index=False)

depois_de_amanha = str(depois_de_amanha())

flt = jogos_do_dia.Date == depois_de_amanha
df_depois_amanha = jogos_do_dia[flt]
df_depois_amanha = df_depois_amanha.sort_values(['Time'])
df_depois_depois_amanha = drop_reset_index(df_depois_amanha)
display(df_depois_amanha)
# df_amanha.to_excel(f"Jogos_do_Dia_Betfair_Back_Lay_{amanha}.xlsx",index=False)

print("Finalizado!")

,League,Date,Time,Home,Away,Odd_H_Back,Odd_H_Lay,Odd_A_Back,Odd_A_Lay,Odd_D_Back,Odd_D_Lay,Odd_Under05_HT_Back,Odd_Under05_HT_Lay,Odd_Over05_HT_Back,Odd_Over05_HT_Lay,Odd_Under15_HT_Back,Odd_Under15_HT_Lay,Odd_Over15_HT_Back,Odd_Over15_HT_Lay,Odd_Under25_HT_Back,Odd_Under25_HT_Lay,Odd_Over25_HT_Back,Odd_Over25_HT_Lay,Odd_Under05_FT_Back,Odd_Under05_FT_Lay,Odd_Over05_FT_Back,Odd_Over05_FT_Lay,Odd_Under15_FT_Back,Odd_Under15_FT_Lay,Odd_Over15_FT_Back,Odd_Over15_FT_Lay,Odd_Under25_FT_Back,Odd_Under25_FT_Lay,Odd_Over25_FT_Back,Odd_Over25_FT_Lay,Odd_Under35_FT_Back,Odd_Under35_FT_Lay,Odd_Over35_FT_Back,Odd_Over35_FT_Lay,Odd_Under45_FT_Back,Odd_Under45_FT_Lay,Odd_Over45_FT_Back,Odd_Over45_FT_Lay,Odd_Under55_FT_Back,Odd_Under55_FT_Lay,Odd_Over55_FT_Back,Odd_Over55_FT_Lay,Odd_Under65_FT_Back,Odd_Under65_FT_Lay,Odd_Over65_FT_Back,Odd_Over65_FT_Lay,Odd_Under75_FT_Back,Odd_Under75_FT_Lay,Odd_Over75_FT_Back,Odd_Over75_FT_Lay,Odd_Under85_FT_Back,Odd_Under85_FT_Lay,Odd_Over85_FT_Back,Odd_Over85_FT_Lay,Odd_BTTS_Yes_Back,Odd_BTTS_Yes_Lay,Odd_BTTS_No_Back,Odd_BTTS_No_Lay,Odd_CS_0x0_Back,Odd_CS_0x0_Lay,Odd_CS_0x1_Back,Odd_CS_0x1_Lay,Odd_CS_0x2_Back,Odd_CS_0x2_Lay,Odd_CS_0x3_Back,Odd_CS_0x3_Lay,Odd_CS_1x0_Back,Odd_CS_1x0_Lay,Odd_CS_1x1_Back,Odd_CS_1x1_Lay,Odd_CS_1x2_Back,Odd_CS_1x2_Lay,Odd_CS_1x3_Back,Odd_CS_1x3_Lay,Odd_CS_2x0_Back,Odd_CS_2x0_Lay,Odd_CS_2x1_Back,Odd_CS_2x1_Lay,Odd_CS_2x2_Back,Odd_CS_2x2_Lay,Odd_CS_2x3_Back,Odd_CS_2x3_Lay,Odd_CS_3x0_Back,Odd_CS_3x0_Lay,Odd_CS_3x1_Back,Odd_CS_3x1_Lay,Odd_CS_3x2_Back,Odd_CS_3x2_Lay,Odd_CS_3x3_Back,Odd_CS_3x3_Lay,Odd_CS_Goleada_H_Back,Odd_CS_Goleada_H_Lay,Odd_CS_Goleada_A_Back,Odd_CS_Goleada_A_Lay,Odd_CS_Goleada_D_Back,Odd_CS_Goleada_D_Lay


,League,Date,Time,Home,Away,Odd_H_Back,Odd_H_Lay,Odd_A_Back,Odd_A_Lay,Odd_D_Back,Odd_D_Lay,Odd_Under05_HT_Back,Odd_Under05_HT_Lay,Odd_Over05_HT_Back,Odd_Over05_HT_Lay,Odd_Under15_HT_Back,Odd_Under15_HT_Lay,Odd_Over15_HT_Back,Odd_Over15_HT_Lay,Odd_Under25_HT_Back,Odd_Under25_HT_Lay,Odd_Over25_HT_Back,Odd_Over25_HT_Lay,Odd_Under05_FT_Back,Odd_Under05_FT_Lay,Odd_Over05_FT_Back,Odd_Over05_FT_Lay,Odd_Under15_FT_Back,Odd_Under15_FT_Lay,Odd_Over15_FT_Back,Odd_Over15_FT_Lay,Odd_Under25_FT_Back,Odd_Under25_FT_Lay,Odd_Over25_FT_Back,Odd_Over25_FT_Lay,Odd_Under35_FT_Back,Odd_Under35_FT_Lay,Odd_Over35_FT_Back,Odd_Over35_FT_Lay,Odd_Under45_FT_Back,Odd_Under45_FT_Lay,Odd_Over45_FT_Back,Odd_Over45_FT_Lay,Odd_Under55_FT_Back,Odd_Under55_FT_Lay,Odd_Over55_FT_Back,Odd_Over55_FT_Lay,Odd_Under65_FT_Back,Odd_Under65_FT_Lay,Odd_Over65_FT_Back,Odd_Over65_FT_Lay,Odd_Under75_FT_Back,Odd_Under75_FT_Lay,Odd_Over75_FT_Back,Odd_Over75_FT_Lay,Odd_Under85_FT_Back,Odd_Under85_FT_Lay,Odd_Over85_FT_Back,Odd_Over85_FT_Lay,Odd_BTTS_Yes_Back,Odd_BTTS_Yes_Lay,Odd_BTTS_No_Back,Odd_BTTS_No_Lay,Odd_CS_0x0_Back,Odd_CS_0x0_Lay,Odd_CS_0x1_Back,Odd_CS_0x1_Lay,Odd_CS_0x2_Back,Odd_CS_0x2_Lay,Odd_CS_0x3_Back,Odd_CS_0x3_Lay,Odd_CS_1x0_Back,Odd_CS_1x0_Lay,Odd_CS_1x1_Back,Odd_CS_1x1_Lay,Odd_CS_1x2_Back,Odd_CS_1x2_Lay,Odd_CS_1x3_Back,Odd_CS_1x3_Lay,Odd_CS_2x0_Back,Odd_CS_2x0_Lay,Odd_CS_2x1_Back,Odd_CS_2x1_Lay,Odd_CS_2x2_Back,Odd_CS_2x2_Lay,Odd_CS_2x3_Back,Odd_CS_2x3_Lay,Odd_CS_3x0_Back,Odd_CS_3x0_Lay,Odd_CS_3x1_Back,Odd_CS_3x1_Lay,Odd_CS_3x2_Back,Odd_CS_3x2_Lay,Odd_CS_3x3_Back,Odd_CS_3x3_Lay,Odd_CS_Goleada_H_Back,Odd_CS_Goleada_H_Lay,Odd_CS_Goleada_A_Back,Odd_CS_Goleada_A_Lay,Odd_CS_Goleada_D_Back,Odd_CS_Goleada_D_Lay
1,German Bundesliga,2024-04-07,10:30:00,Hoffenheim,Augsburg,2.32,2.34,3.05,3.10,4.1,4.2,3.60,5.4,1.22,1.38,1.72,1.82,2.20,2.40,1.17,1.30,4.40,6.8,22.0,36.0,1.03,1.05,6.4,7.8,1.16,1.18,2.80,2.98,1.51,1.56,1.78,1.82,2.22,2.30,1.32,1.35,3.85,4.10,1.14,1.16,7.4,8.2,1.05,1.07,16.5,21.0,1.01,1.03,38.0,110.0,1.01,1.01,100.0,1000.0,1.50,1.53,2.92,3.00,23.0,28.0,19.0,22.0,24.0,27.0,42.0,50.0,16.0,18.0,9.6,10.5,12.5,13.5,24.0,27.0,18.0,20.0,11.0,12.0,13.0,14.0,26.0,30.0,29.0,32.0,18.5,20.0,22.0,25.0,42.0,48.0,9.2,9.8,13.5,15.0,42.0,330.0
2,English Premier League,2024-04-07,11:30:00,Man Utd,Liverpool,4.90,5.00,1.68,1.69,4.8,4.9,4.20,6.4,1.22,1.26,1.90,2.04,1.97,2.12,1.28,1.32,4.10,4.8,34.0,50.0,1.02,1.03,8.6,9.2,1.12,1.13,3.55,3.65,1.38,1.39,2.10,2.14,1.89,1.90,1.47,1.49,3.00,3.15,1.21,1.24,5.3,5.8,1.08,1.10,11.0,13.0,1.03,1.05,23.0,36.0,1.01,1.02,55.0,110.0,1.46,1.47,3.10,3.20,36.0,38.0,18.0,18.5,15.0,16.0,19.5,20.0,34.0,36.0,13.0,13.5,10.5,11.0,13.0,13.5,48.0,50.0,20.0,22.0,15.0,15.5,19.5,20.0,95.0,110.0,36.0,44.0,32.0,36.0,40.0,44.0,24.0,25.0,4.6,4.8,140.0,210.0
3,German Bundesliga,2024-04-07,12:30:00,Wolfsburg,Mgladbach,1.93,1.94,4.20,4.30,4.1,4.2,3.40,4.2,1.31,1.41,1.52,1.63,2.60,2.90,1.15,1.20,6.00,7.8,17.5,21.0,1.05,1.07,5.0,5.8,1.21,1.25,2.38,2.42,1.71,1.73,1.55,1.59,2.72,2.82,1.22,1.24,5.20,5.50,1.08,1.10,11.0,14.5,1.02,1.04,26.0,55.0,1.01,1.02,70.0,1000.0,1.01,1.01,190.0,1000.0,1.66,1.73,2.38,2.50,18.5,19.5,18.0,20.0,30.0,34.0,60.0,85.0,11.0,12.0,8.6,9.2,15.5,16.5,40.0,44.0,12.0,13.5,9.6,10.5,15.5,16.5,42.0,48.0,20.0,22.0,16.0,17.5,26.0,28.0,55.0,75.0,9.2,9.6,30.0,36.0,22.0,980.0
4,English Premier League,2024-04-07,13:30:00,Sheff Utd,Chelsea,7.80,8.00,1.42,1.44,5.7,5.8,3.85,5.9,1.20,1.35,1.80,1.91,2.10,2.26,1.22,1.30,4.30,5.6,28.0,50.0,1.03,1.04,7.4,8.0,1.14,1.16,3.20,3.35,1.43,1.46,1.90,1.93,2.08,2.10,1.38,1.41,3.45,3.65,1.15,1.18,6.6,7.8,1.07,1.08,13.5,18.0,1.02,1.04,26.0,55.0,1.01,1.02,60.0,1000.0,1.63,1.67,2.48,2.58,32.0,34.0,13.5,14.5,11.0,11.5,13.0,14.5,36.0,40.0,12.5,13.5,9.8,10.0,11.5,12.5,50.0,75.0,26.0,28.0,18.5,19.5,22.0,23.0,120.0,210.0,70.0,80.0,60.0,65.0,60.0,70.0,55.0,65.0,4.1,4.3,190.0,980.0
5,English Premier League,2024-04-07,14:00:00,Tottenham,Nottm Forest,1.41,1.42,8.40,8.60,5.8,5.9,1.01,1000.0,1.01,1000.00,1.02,1000.00,1.68,3.20,1.01,1000.00,1.0

,League,Date,Time,Home,Away,Odd_H_Back,Odd_H_Lay,Odd_A_Back,Odd_A_Lay,Odd_D_Back,Odd_D_Lay,Odd_Under05_HT_Back,Odd_Under05_HT_Lay,Odd_Over05_HT_Back,Odd_Over05_HT_Lay,Odd_Under15_HT_Back,Odd_Under15_HT_Lay,Odd_Over15_HT_Back,Odd_Over15_HT_Lay,Odd_Under25_HT_Back,Odd_Under25_HT_Lay,Odd_Over25_HT_Back,Odd_Over25_HT_Lay,Odd_Under05_FT_Back,Odd_Under05_FT_Lay,Odd_Over05_FT_Back,Odd_Over05_FT_Lay,Odd_Under15_FT_Back,Odd_Under15_FT_Lay,Odd_Over15_FT_Back,Odd_Over15_FT_Lay,Odd_Under25_FT_Back,Odd_Under25_FT_Lay,Odd_Over25_FT_Back,Odd_Over25_FT_Lay,Odd_Under35_FT_Back,Odd_Under35_FT_Lay,Odd_Over35_FT_Back,Odd_Over35_FT_Lay,Odd_Under45_FT_Back,Odd_Under45_FT_Lay,Odd_Over45_FT_Back,Odd_Over45_FT_Lay,Odd_Under55_FT_Back,Odd_Under55_FT_Lay,Odd_Over55_FT_Back,Odd_Over55_FT_Lay,Odd_Under65_FT_Back,Odd_Under65_FT_Lay,Odd_Over65_FT_Back,Odd_Over65_FT_Lay,Odd_Under75_FT_Back,Odd_Under75_FT_Lay,Odd_Over75_FT_Back,Odd_Over75_FT_Lay,Odd_Under85_FT_Back,Odd_Under85_FT_Lay,Odd_Over85_FT_Back,Odd_Over85_FT_Lay,Odd_BTTS_Yes_Back,Odd_BTTS_Yes_Lay,Odd_BTTS_No_Back,Odd_BTTS_No_Lay,Odd_CS_0x0_Back,Odd_CS_0x0_Lay,Odd_CS_0x1_Back,Odd_CS_0x1_Lay,Odd_CS_0x2_Back,Odd_CS_0x2_Lay,Odd_CS_0x3_Back,Odd_CS_0x3_Lay,Odd_CS_1x0_Back,Odd_CS_1x0_Lay,Odd_CS_1x1_Back,Odd_CS_1x1_Lay,Odd_CS_1x2_Back,Odd_CS_1x2_Lay,Odd_CS_1x3_Back,Odd_CS_1x3_Lay,Odd_CS_2x0_Back,Odd_CS_2x0_Lay,Odd_CS_2x1_Back,Odd_CS_2x1_Lay,Odd_CS_2x2_Back,Odd_CS_2x2_Lay,Odd_CS_2x3_Back,Odd_CS_2x3_Lay,Odd_CS_3x0_Back,Odd_CS_3x0_Lay,Odd_CS_3x1_Back,Odd_CS_3x1_Lay,Odd_CS_3x2_Back,Odd_CS_3x2_Lay,Odd_CS_3x3_Back,Odd_CS_3x3_Lay,Odd_CS_Goleada_H_Back,Odd_CS_Goleada_H_Lay,Odd_CS_Goleada_A_Back,Odd_CS_Goleada_A_Lay,Odd_CS_Goleada_D_Back,Odd_CS_Goleada_D_Lay


Finalizado!
